# 10 | Raw-IMU Temporal Classification with PyTorch

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Train an interpretable, compact 1D-CNN directly on six-channel, 100-point HARTH acceleration windows. Contrast raw-sequence learning with previous feature-engineered baselines conceptually, without claiming a matched performance benchmark.

**Scope:** An intentionally bounded, deterministic subset (at most 500 eligible windows per participant). Train and Validation subjects are independent. Original reserved Test IDs S023 and S032 are never loaded. Results must NOT be compared numerically to the full 132,397-window, five-fold classical ML experiment as if protocols were identical.

## 01. Imports and research environment

The real model, scaling and training logic remain in `.py` files. Code-cell comments describe exactly why each operation is needed.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# 1. Locate the repository and register the existing research modules.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Run inside the human-movement-intelligence repository")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

# 2. Reuse source decoding, time-safe windows and grouped splitting.
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_window_cohort
from core.data_splitting import split_group_indices
from core.temporal_cnn import TemporalCNN, train_temporal_cnn

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Python environment:", sys.executable)

## 02. Define the bounded raw-window experiment

Use all **29 development identities** with a cap of **500 windows per person** selected evenly along each source recording, without label-based sampling. This cap is purely computational and may change the activity distribution. It does not create a statistically balanced dataset.

In [ ]:
# 1. Keep the original final Test subjects outside data loading.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
MAX_WINDOWS_PER_SUBJECT = 500
assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# 2. Reuse the same source timestamp and annotation-purity criteria.
X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    max_windows_per_subject=MAX_WINDOWS_PER_SUBJECT,
    window_size=100, step_size=50, sampling_rate=50,
)
display(overview)
print("Real raw IMU tensor:", X.shape)
print("Label vector:", y.shape)
print("Subjects:", metadata["subject_id"].nunique())

## 03. Subject-disjoint Training and Validation only

Rather than introducing a new random window split, reserve entire participants through the existing deterministic grouped split. The third partition is **not used** here, and is named `unused_development` to distinguish it from the original S023/S032 final Test.

In [ ]:
# 1. Allocate independent development participants with a fixed seed.
parts = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
group_names = {
    name: sorted(metadata.iloc[idx]["subject_id"].unique().tolist())
    for name, idx in parts.items()
}
assert not (set(group_names["train"]) & set(group_names["validation"]))
assert not (set(group_names["train"]) & set(group_names["test"]))
assert not (set(group_names["validation"]) & set(group_names["test"]))

# 2. Use ONLY Train and Validation windows for this development run.
train_rows = parts["train"]
validation_rows = parts["validation"]
X_train, y_train = X[train_rows], y[train_rows]
X_validation, y_validation = X[validation_rows], y[validation_rows]
display(pd.DataFrame([
    {"role": "Train", "subjects": len(group_names["train"]),
     "windows": len(train_rows), "subject_ids": ", ".join(group_names["train"])},
    {"role": "Validation", "subjects": len(group_names["validation"]),
     "windows": len(validation_rows), "subject_ids": ", ".join(group_names["validation"])},
    {"role": "Unused development", "subjects": len(group_names["test"]),
     "windows": len(parts["test"]), "subject_ids": ", ".join(group_names["test"])},
]))
print("Original reserved final Test loaded: NO")

## 04. Audit participant/class counts and the model input

Before fitting, verify the selected Train group still contains all eight target classes. The CNN processes `(batch, channels, time)`, not the handcrafted 30-feature matrix.

In [ ]:
# 1. Summarise source activity counts for the actual training and validation windows.
counts = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(),
    "validation": pd.Series(y_validation).value_counts(),
}).reindex(TASK_CLASSES, fill_value=0).fillna(0).astype(int)
counts.index = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
display(counts)
assert (counts["train"] > 0).all(), "Training class coverage is incomplete."

# 2. Verify tensor shapes before training the temporal architecture.
model_preview = TemporalCNN(in_channels=6, n_classes=8)
example_output = model_preview(torch.from_numpy(X_train[:4]))
print("Example CNN output shape:", tuple(example_output.shape))
print("Untrained classifier output is logits, not probabilities.")

## 05. Train a compact 1D-CNN with leakage-aware scaling

Channel means and standard deviations are calculated **only on Train**. The model uses convolutional filters over time; there is no handcrafted feature extraction. CPU is the default, with CUDA optional when available. Results are exploratory and depend on the bounded subset.

In [ ]:
# 1. Keep a modest training budget for the first functional PyTorch baseline.
EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

# 2. Fit fresh CNN weights and evaluate on independent Validation subjects.
model, history, validation_pred, scaling = train_temporal_cnn(
    X_train=X_train, y_train=y_train,
    X_validation=X_validation, y_validation=y_validation,
    classes=TASK_CLASSES,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, seed=42, device=DEVICE,
)
display(history.round(4))
print("Channel means fitted from Train:", np.round(scaling["channel_mean"], 4))

## 06. Visualise actual training progression

Train loss and Validation Macro F1 answer different questions and therefore use separate panels. The last epoch is the reported result; no final Test prediction or benchmark score is generated.

In [ ]:
# 1. Plot training loss for every epoch.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[0].grid(alpha=0.25)

# 2. Plot held-out-participant Validation Macro F1 independently.
axes[1].plot(history["epoch"], history["validation_macro_f1"], marker="o")
axes[1].set(title="Validation Macro F1", xlabel="Epoch",
            ylabel="Macro F1", ylim=(0, 1))
axes[1].grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 07. Inspect the last-epoch predictions by activity

The confusion matrix shows which activity codes the temporal model confuses. This is a descriptive diagnostic on the bounded development Validation sample, not a final test.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

# 1. Show precision, recall, F1 and sample support in the fixed class order.
report = classification_report(
    y_validation, validation_pred,
    labels=TASK_CLASSES,
    target_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    output_dict=True, zero_division=0,
)
display(pd.DataFrame(report).T.round(3))

# 2. Display the real held-out Validation confusion matrix.
confusion = confusion_matrix(y_validation, validation_pred, labels=TASK_CLASSES)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
).plot(ax=ax, cmap="Blues", xticks_rotation=40, colorbar=False)
ax.set_title("Raw IMU 1D-CNN: bounded development Validation")
fig.tight_layout()
plt.show()

## 08. Research quality gate and limitations

Confirm strict subject separation, valid class predictions and training history. This stage proves the raw-signal PyTorch pipeline runs; it does **not** establish that deep learning outperforms feature-engineered Random Forest, because dataset sampling and validation protocols differ.

In [ ]:
# 1. Check subject identity isolation and tensor/label correspondence.
assert len(X_train) == len(y_train) and len(X_validation) == len(y_validation)
assert X_train.shape[1:] == X_validation.shape[1:] == (6, 100)
assert not (set(group_names["train"]) & set(group_names["validation"]))
assert set(np.unique(validation_pred)).issubset(TASK_CLASSES)
assert history["validation_macro_f1"].between(0, 1).all()
assert len(history) == EPOCHS

# 2. State precisely what was and was not evaluated.
print("Raw-signal PyTorch pipeline: PASSED")
print("Validated development subjects:", len(group_names["validation"]))
print("Unused development subjects:", len(group_names["test"]))
print("Original reserved Test loaded/scored: NO")
print("Matched full-cohort CNN vs Random Forest benchmark: NOT PERFORMED")